In [13]:
import pandas as pd

df = pd.read_csv('calls_puts_spx_bloom.csv')

print(df.head())
print(df.info())

   Strike             Ticker          Bid          Ask  Last        IVM  Volm  \
0    4750  SPX 3/19/27 C4750  2319.500000  2331.300049   0.0  29.152979     0   
1    4775  SPX 3/19/27 C4775  2296.899902  2308.600098   0.0  28.985453     0   
2    4800  SPX 3/19/27 C4800  2275.600098  2287.199951   0.0  28.821543     0   
3    4825  SPX 3/19/27 C4825  2252.199951  2263.699951   0.0  28.668737     0   
4    4850  SPX 3/19/27 C4850  2230.800049  2242.199951   0.0  28.533421     0   

   Strike.1           Ticker.1      Bid.1      Ask.1      Last.1      IVM.1  \
0      4750  SPX 3/19/27 P4750  80.800003  81.599998   86.000000  30.007044   
1      4775  SPX 3/19/27 P4775  82.400002  83.199997   90.300003  29.856182   
2      4800  SPX 3/19/27 P4800  84.000000  84.800003  104.550003  29.699228   
3      4825  SPX 3/19/27 P4825  85.599998  86.400002   93.199997  29.543734   
4      4850  SPX 3/19/27 P4850  87.400009  88.099991   91.599998  29.393513   

   Volm.1  
0       0  
1       0  
2 

In [14]:
import sys
import numpy as np
from scipy.optimize import minimize

from riccati import solve_riccati
from rough_heston import rough_heston_joint_mse

In [15]:
# call mid prices
df['Mid_Call'] = (df['Bid'] + df['Ask']) / 2
calls_data = df[['Strike', 'Bid', 'Ask', 'Mid_Call']].copy()

# put mid prices
df['Mid_Put'] = (df['Bid.1'] + df['Ask.1']) / 2
puts_data = df[['Strike.1', 'Bid.1', 'Ask.1', 'Mid_Put']].copy()

# just renaming columns for clarity
calls_data.columns = ['Strike', 'Bid', 'Ask', 'Mid_Call']
puts_data.columns = ['Strike', 'Bid', 'Ask', 'Mid_Put']

calls_data.head()

,Strike,Bid,Ask,Mid_Call
0,4750,2319.500000,2331.300049,2325.400024
1,4775,2296.899902,2308.600098,2302.750000
2,4800,2275.600098,2287.199951,2281.400024
3,4825,2252.199951,2263.699951,2257.949951
4,4850,2230.800049,2242.199951,2236.500000


In [16]:
puts_data.head()

,Strike,Bid,Ask,Mid_Put
0,4750,80.800003,81.599998,81.200001
1,4775,82.400002,83.199997,82.799999
2,4800,84.000000,84.800003,84.400002
3,4825,85.599998,86.400002,86.000000
4,4850,87.400009,88.099991,87.750000


In [17]:
# initial param guess from the paper
initial_guess = [0.6, 1.0, -0.5, 0.05, 0.04, 0.04]

# explicit bounds that we will pass to the optimizer
bounds = [
    (0.51, 0.99),  # alpha (roughness)
    (0.1, 5.0),    # lambda (mean reversion speed)
    (-0.9, 0.7),   # rho (correlation / skew)
    (1e-4, 0.5),   # nu (vol‑of‑vol)
    (1e-4, 0.2),   # theta (level)
    (1e-4, 1.0)    # V0 (initial variance)
]

market_strikes = df['Strike'].values
S0 = 6876.3
r = 0.045
T = 1.0

# helper that prints progress occasionally
def callback(xk):
    if np.random.rand() < 0.01:  # print roughly 1% of the evaluations
        mse = rough_heston_joint_mse(xk, market_strikes, call_market_prices, market_strikes, put_market_prices, S0, T, r)
        print("iter", xk, "mse", mse)

# quick sanity check at the starting point
print("initial mse (calls)", rough_heston_joint_mse(initial_guess, market_strikes, calls_data['Mid_Call'].values, market_strikes, puts_data['Mid_Put'].values, S0, T, r))
print("initial mse (puts)", rough_heston_joint_mse(initial_guess, market_strikes, puts_data['Mid_Put'].values, market_strikes, calls_data['Mid_Call'].values, S0, T, r))

call_market_prices = calls_data['Mid_Call'].values
put_market_prices = puts_data['Mid_Put'].values

initial mse (calls) 5239.462653506308
initial mse (puts) 2308021.4884630833


In [18]:
# Extract the underlying data from your dataframes
call_strikes = calls_data['Strike'].values
call_prices = calls_data['Mid_Call'].values

put_strikes = puts_data['Strike'].values
put_prices = puts_data['Mid_Put'].values

In [19]:
result_joint = minimize(
    rough_heston_joint_mse,
    initial_guess,
    args=(call_strikes, call_prices, put_strikes, put_prices, S0, T, r),
    method='L-BFGS-B',
    bounds=bounds,
    options={'ftol': 1e-9, 'disp': True, 'maxiter': 150},
    callback=callback
)


/var/folders/x4/hm69h0zd38z3_hl9mbn6p12h0000gn/T/ipykernel_71369/1404713126.py:1: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result_joint = minimize(
/Users/arthur/Documents/Market Microstructure/riccati.py:8: RuntimeWarning: overflow encountered in scalar power
  term3 = 0.5 * (lambd * nu)**2 * x**2
/Users/arthur/Documents/Market Microstructure/riccati.py:8: RuntimeWarning: invalid value encountered in scalar power
  term3 = 0.5 * (lambd * nu)**2 * x**2
/Users/arthur/Documents/Market Microstructure/riccati.py:8: RuntimeWarning: invalid value encountered in scalar multiply
  term3 = 0.5 * (lambd * nu)**2 * x**2
/Users/arthur/Documents/Market Microstructure/riccati.py:8: RuntimeWarning: overflow encountered in scalar multiply
  term3 = 0.5 * (lambd * nu)**2 * x**2
/Users/arthur/Documents/Market Microstructure/riccati.py:62: RuntimeWarning: invalid value encountered in multiply
  h_predi

In [20]:
print(result_joint.x)

[ 0.5688053   2.38538217 -0.63879859  0.20700209  0.03968418  0.03291164]


In [ ]:
# old [ 0.56814973  2.41570854 -0.64205659  0.21004004  0.03466721  0.03133643]

In [ ]:
# new [ 0.5688053   2.38538217 -0.63879859  0.20700209  0.03968418  0.03291164]